# 59. The shared direction, with the wording taken out, across the trained defences

**Where this starts.** Notebooks 57f and 58 reported one result on all three models: the heads respond to an injected instruction in the same direction as to the user's own instruction (median cosine on the carrier heads 0.73 and 0.88 on Qwen 3B, 0.80 and 0.86 on Qwen 7B, 0.83 and 0.85 on Llama 8B, at the template positions and the answer tokens). The 20-head localisation held only on the 3B. Part two's training loss is the one SecOPD published (Peng, Lian, Wagner and Chen, arXiv 2608.21500), and the deletion problem its literal phase fixes is reported by Hermon et al. (ICML 2026, arXiv 2606.30783); neither explains why a trained defence costs the user's own instructions. This notebook asks two things. First, whether the shared direction survives a control those notebooks did not have: the same words in the two places. Second, if it does, whether it tracks what each defence this project has already trained achieves and what it costs. No training; one forward pass per prompt.

**Why the control is needed.** Both directions in 57f and 58 compare a prompt that holds an instruction with one that does not, and the instruction's words are much the same in the two. A model that knew nothing about instructions would still respond alike to the same words in two places: while building this notebook, small networks with random weights gave a cosine of about 0.53 to 0.77 between the two responses on synthetic text, for an instruction and for an ordinary sentence alike. So this notebook compares an instruction with an ordinary sentence of about the same length in the same places, and compares each base model with its own architecture with random weights reading the same prompts.

**What is read, in plain words.** For each of the 210 injections, five prompts that differ in one sentence only:

| prompt | the extra sentence | where |
|---|---|---|
| `P` | none (the clean document) | |
| `D_ins` | the injected instruction (the document as it was attacked) | last line of the document |
| `D_stmt` | an ordinary sentence of about the same length, from a SQuAD validation paragraph (Wikipedia), where notebook 37 took its statements; the BIPIA hosts hold too few clean prose sentences | the same place |
| `U_ins` | the same instruction | in the user turn, after the task sentence (with Meta-SecAlign's input role, the task sentence and this sentence stay in the user role and the document goes in the input role) |
| `U_stmt` | the same ordinary sentence | the same place in the user turn |

Every head's output is read in every layer over 21 positions: the five template positions where the answer starts, then 16 answer tokens appended after them. The 16 tokens are the same for all five prompts of an item (the base model's own greedy start of an answer to `D_stmt`), so the prompts of an item differ only in the one sentence and where it sits.

- **Sharedness** (the shared direction with the wording taken out), per item: the cosine between the change the instruction makes in the document (`D_ins` minus `P`) and the change the same instruction makes in the user turn (`U_ins` minus `P`), minus the same cosine for the ordinary sentence (`D_stmt` minus `P` against `U_stmt` minus `P`). The same words sit in both places either way. Above 0 means the model treats an instruction in the document more like the same instruction from the user than it treats an ordinary sentence in the two places. Sentence length and form can still move it, and they move it differently in every random network (while building this notebook, five random draws of one small architecture gave values from +0.01 to -0.24). So each base model is set against five seeded draws of its own architecture with random weights, which read the same prompts with the same answer tokens on 105 of the items; what a pretrained model shows beyond all five it has learned.
- **User drive**: how far `U_ins` minus `U_stmt` moves the heads along a user-instruction direction built in the undefended base model from the other half of the host documents and from the other six kinds of task only. It measures how strongly a model registers an instruction from its user. It also carries some wording, which is the same for every model of a family, so the marks read its change from the base model, never its level.
- **Injected drive** (reported): the same for `D_ins` minus `D_stmt`.

Every defended model is read in its base model's head coordinates (every adapter here, Meta-SecAlign-8B included, is a low-rank adapter on those base weights). Intervals are 95% bootstrap intervals over host documents.

**The conditions.** Qwen 3B: undefended, twin distillation v1, v2, v3, distillation without twins at two scales, the full recipe v4, and the three literal-phase controls of notebook 50. Qwen 7B: undefended, distillation without twins, v4 and v5 (the final 7B). Llama 8B: undefended, our phase 1 and full recipe, and Meta-SecAlign-8B read twice, with its input role as its authors specify and with the document in the user turn. 19 conditions on 18 sets of weights.

**What each defence achieves and costs** is read from the repository's tables, not measured again: attack success with 13 fixed tries under the paper's rule (`reports/paper_fixed_all_models.csv`, and `controls_security.csv` for the three controls of notebook 50, which uses the same rule; all 19 conditions); user instructions followed on the 150 held-out twins (`controls_utility.csv`, `ablation_anchor_scale.csv`, `fidelity_v5_7b_utility.csv`, `llama_utility.csv`, 18 conditions); the deletion gap of notebook 37 (`controls_fidelity.csv`, `fidelity_v5_7b_fidelity.csv`, `llama_fidelity.csv`), reported only.

**Marks, fixed in Stage A before any result.**

| mark | what must hold |
|---|---|
| V: the shared direction is about instructions | on each of the three undefended models, sharedness on the 105 items at least 0.05 above the highest of its architecture's five random-weight draws. If it fails, the shared direction of 57f and 58 was the wording, and nothing else is read. |
| H1: it tracks what a defence achieves | across the 16 trained conditions, the change in sharedness (condition minus its base) and the change in attack success: Spearman rho at least 0.50, and a one-sided permutation p below 0.05 with the changes shuffled only within each model family, so that a shift shared by a whole family cannot make the result. |
| H2: the user side tracks what a defence costs | across the 15 trained conditions with the 150-twin measure, the change in user drive and the change in user instructions followed: rho at least 0.50, p below 0.05, shuffled the same way. Meta-SecAlign-8B enters H1 and H2 twice (one set of weights, two prompt formats). |
| H3: the same weights, two formats | Meta-SecAlign-8B with its input role against the same weights with the document in the user turn (attack success 0.062 against 0.195): sharedness lower with the input role, the paired interval's upper end below 0. |
| decision | V, H1 and H2 hold: GO, the next notebook trains against sharedness while holding the user drive. V and H1 hold, H2 fails: sharedness is the training target, and the user cost is measured, not predicted. V fails: CLOSE, the shared direction was the wording. H1 fails: CLOSE, the shared direction is not what trained defences change, and this route gives part two no method. H3 is reported in every case. |

**Compute.** 19 conditions times 1,050 prompts (53 forward passes of four items, twenty prompts each), three times 210 short generations for the answer tokens, 420 prompts per base model for its user directions, and five draws of each architecture with random weights (bf16, built on the GPU, one at a time) on 525 prompts each. The Hugging Face login and the tokenizer checks for Qwen 7B, Llama and Meta-SecAlign run at the end of Stage B, before the long part. The two Qwen families run in 4-bit, Llama and Meta-SecAlign in bf16 as in notebooks 41 to 45. Every condition is cached as soon as it is read, and the notebook resumes. Llama and Meta-SecAlign need the Hugging Face login the project's helper asks for (the Colab secret `HF_TOKEN`, or a hidden box in the cell output).

**Outputs.** `reports/shared_drive/drive.csv`, `layers.csv`, `statements.csv`, `go_criteria_59.csv`, `go_59.txt`; `figures/shared_drive/drive.png`.

In [ ]:
# === SESSION BOOTSTRAP (run first): install from the local disk, then mount ===
!pip install -q transformers datasets accelerate sentencepiece bitsandbytes peft langdetect==1.0.9 scikit-learn
!pip uninstall -y -q torchao   # Colab ships an old torchao that the current PEFT rejects; nothing in this project uses it
from google.colab import drive
drive.mount('/content/drive')
import os, sys
DRIVE_ROOT = '/content/drive/MyDrive/PICALIB_Research'
REPO_DIR   = os.path.join(DRIVE_ROOT, 'picalib-research')
!git config --global user.name  "Md Anas Biswas"
!git config --global user.email "anasbiswas@gmail.com"
!git config --global credential.helper "store --file={DRIVE_ROOT}/.git-credentials"
%cd {REPO_DIR}
sys.path.insert(0, 'src')
!git pull
for d_ in ('reports/shared_drive', 'figures/shared_drive', 'data/shared_drive'): os.makedirs(d_, exist_ok=True)
import torch, gc
assert torch.cuda.is_available(), 'this notebook needs a GPU'
print('Session ready. GPU:', torch.cuda.get_device_name(0))

In [ ]:
%%writefile src/shared_drive.py
"""How strongly an instruction pushes a model along the user's instruction direction, read from every attention head at
the positions where the answer starts and over the first answer tokens. Built on task_state and head_patching, which
are unchanged; nothing in the model is changed.

Reading. Forward pre-hooks on every layer's attention output projection (o_proj) read the heads' outputs (the
projection's input) over a window of W positions at the end of each row: the K template positions where the answer
starts, then the answer tokens appended after them (teacher forcing, the same tokens for every prompt of an item). One
forward pass of the decoder per batch, with position ids taken from the attention mask as generation computes them,
and no language-model head. For every row the hooks keep:
  sums  the running sum of the window per group of rows (the caller names a group for every row), [L, W, nh*dh] float32;
  proj  when reference directions are set, the row's dot product with each reference, layer by layer and separately
        for the template part and the answer part of the window: [rows, L, R, 2];
  pairs for pairs of differences between rows of the same batch, (a - c, b - d), the dot product <a - c, b - d> and the
        two squared sizes, layer by layer and per part: [pairs, L, 3, 2]. This is how the same sentence in two places is
        compared item by item.

Drive. With u a reference direction (in the base model, the fold mean of an instruction in the user turn minus an
ordinary sentence of the same length in the same place), the drive of a prompt pair (a, b) on an item is
<x_a - x_b, u> / <u, u> over the window, with u taken from the other fold of items (cross-fitting, so an item's own
noise never sits in its own reference). 1.0 is the push a user instruction gives in the base model; 0 is none.
"""
from __future__ import annotations
import numpy as np
from task_state import decoder, decoder_layers
from head_patching import head_dims


def encode_convs(tok, convs, sfx, ans_ids=None, max_length=4096):
    """Left-padded batch of conversations (message lists), rendered by the tokenizer's own chat template with the
    generation prompt and tokenized without adding special tokens again (the template already carries them), as the
    project's generation calls do. Every row must end with the template suffix sfx; ans_ids (one list per row, all the
    same length) are appended after it. Returns (input_ids, attention_mask) on the CPU."""
    import torch
    rows = []
    for c in convs:
        r = list(tok(tok.apply_chat_template(c, tokenize=False, add_generation_prompt=True), add_special_tokens=False)["input_ids"])
        assert r[-len(sfx):] == list(sfx), "a prompt does not end with the template suffix"
        rows.append(r)
    if ans_ids is not None:
        assert len(ans_ids) == len(rows) and len({len(a) for a in ans_ids}) == 1, "answer tokens: one list per row, all the same length"
        rows = [r + list(a) for r, a in zip(rows, ans_ids)]
    T = max(len(r) for r in rows); assert T <= max_length, f"a prompt has {T} tokens"
    ids = torch.full((len(rows), T), int(tok.pad_token_id), dtype=torch.long); am = torch.zeros((len(rows), T), dtype=torch.long)
    for j, r in enumerate(rows):
        ids[j, T - len(r):] = torch.tensor(r, dtype=torch.long); am[j, T - len(r):] = 1
    return ids, am


class DriveHooks:
    def __init__(self, model, W, K):
        self.layers = decoder_layers(model); self.L = len(self.layers); self.nh, self.dh = head_dims(model); self.D = self.nh * self.dh
        self.W, self.K = int(W), int(K); assert self.W >= self.K >= 1
        self.active = False; self.group = None; self.sums = {}; self.refs = None; self.proj = None; self.raw = None; self.passes = 0
        self.pidx = None; self.pd = None                    # local row indices of the batch's pairs [4, P] and their output [P, L, 3, 2]
        self.handles = [l.self_attn.o_proj.register_forward_pre_hook(self._make(i)) for i, l in enumerate(self.layers)]

    def set_refs(self, refs):
        """refs: [R, L, W, D] float32 (or None). Kept on the model's device."""
        import torch
        dev = self.layers[0].self_attn.o_proj.weight.device
        self.refs = None if refs is None else torch.as_tensor(refs, dtype=torch.float32).to(dev)
        if self.refs is not None: assert tuple(self.refs.shape[1:]) == (self.L, self.W, self.D), (tuple(self.refs.shape), (self.L, self.W, self.D))

    def _make(self, i):
        def hook(module, args):
            import torch
            if not self.active: return None
            x = args[0]
            if not isinstance(x, torch.Tensor) or x.dim() != 3 or x.shape[1] < self.W: return None
            if i == 0: self.passes += 1
            t = x[:, -self.W:, :].float()                                           # [B, W, D]
            for g, rows in self.group.items():
                s = self.sums.get(g)
                if s is None: s = self.sums[g] = torch.zeros(self.L, self.W, self.D, dtype=torch.float32, device=x.device)
                s[i] += t[rows].sum(0)
            if self.refs is not None:
                r = self.refs[:, i]                                                 # [R, W, D]
                self.proj[:, i, :, 0] = torch.einsum('bkd,rkd->br', t[:, :self.K], r[:, :self.K])
                if self.W > self.K: self.proj[:, i, :, 1] = torch.einsum('bkd,rkd->br', t[:, self.K:], r[:, self.K:])
            if self.pidx is not None:
                a = t[self.pidx[0]] - t[self.pidx[2]]; b = t[self.pidx[1]] - t[self.pidx[3]]                # [P, W, D]
                for part, sl in ((0, slice(0, self.K)), (1, slice(self.K, self.W))):
                    if sl.stop <= sl.start: continue
                    aa, bb = a[:, sl], b[:, sl]
                    self.pd[:, i, 0, part] = (aa * bb).sum((1, 2)); self.pd[:, i, 1, part] = (aa * aa).sum((1, 2)); self.pd[:, i, 2, part] = (bb * bb).sum((1, 2))
            if self.raw is not None: self.raw[i] = t.detach().cpu()
            return None
        return hook

    def close(self):
        for h in self.handles: h.remove()
        self.handles = []


def read(model, hooks, tok, convs, groups, sfx, ans_ids=None, batch=16, order=None, keep_raw=False, batches=None, pairs=None):
    """One decoder pass per batch. groups: a group name per conversation (sums are kept per group in hooks.sums).
    batches: explicit lists of row indices, one per forward pass (default: consecutive chunks of order). pairs: tuples of
    rows (a, b, c, d); every pair must sit inside one batch. Returns (proj [n, L, R, 2] or None without references,
    counts per group, raw window per row or None, pair results [len(pairs), L, 3, 2] or None)."""
    import torch
    n = len(convs); idx = list(order) if order is not None else list(range(n)); assert sorted(idx) == list(range(n))
    if batches is None: batches = [idx[s: s + batch] for s in range(0, n, batch)]
    assert sorted(r for b in batches for r in b) == list(range(n)), "the batches must hold every row once"
    R = 0 if hooks.refs is None else int(hooks.refs.shape[0])
    out = np.zeros((n, hooks.L, R, 2), np.float32) if R else None; cnt = {}; raws = {} if keep_raw else None
    pout = None if pairs is None else np.full((len(pairs), hooks.L, 3, 2), np.nan, np.float32)
    bof = {r: k for k, b in enumerate(batches) for r in b}; pin = {}
    for q, pr in enumerate(pairs or []):
        ks = {bof[r] for r in pr}; assert len(ks) == 1, f"pair {q} spans more than one batch"; pin.setdefault(ks.pop(), []).append(q)
    dec = decoder(model); dev = hooks.layers[0].self_attn.o_proj.weight.device
    hooks.active = True
    try:
        for k, ch in enumerate(batches):
            ch = list(ch)
            ids, am = encode_convs(tok, [convs[i] for i in ch], sfx, None if ans_ids is None else [ans_ids[i] for i in ch])
            assert ids.shape[1] >= hooks.W, "a row is shorter than the window"
            emb_dev = dec.embed_tokens.weight.device; ids, am = ids.to(emb_dev), am.to(emb_dev)
            pos = (am.cumsum(-1) - 1).clamp_min(0)                                 # as generation computes them for left padding
            gm = {}
            for j, i in enumerate(ch): gm.setdefault(groups[i], []).append(j)
            hooks.group = {g: torch.tensor(r, dtype=torch.long, device=dev) for g, r in gm.items()}
            for g, r in gm.items(): cnt[g] = cnt.get(g, 0) + len(r)
            if R: hooks.proj = torch.zeros(len(ch), hooks.L, R, 2, dtype=torch.float32, device=dev)
            if keep_raw: hooks.raw = [None] * hooks.L
            qs = pin.get(k, [])
            if qs:
                loc = {r: j for j, r in enumerate(ch)}
                hooks.pidx = torch.tensor([[loc[pairs[q][z]] for q in qs] for z in range(4)], dtype=torch.long, device=dev)
                hooks.pd = torch.zeros(len(qs), hooks.L, 3, 2, dtype=torch.float32, device=dev)
            else: hooks.pidx = None; hooks.pd = None
            before = hooks.passes
            with torch.no_grad(): dec(input_ids=ids, attention_mask=am, position_ids=pos, use_cache=False)
            assert hooks.passes == before + 1, "the hooks did not see this forward pass"
            if R: out[ch] = hooks.proj.cpu().numpy()
            if qs: pout[qs] = hooks.pd.cpu().numpy()
            if keep_raw:
                for j, i in enumerate(ch): raws[i] = torch.stack([hooks.raw[l][j] for l in range(hooks.L)])     # [L, W, D]
    finally:
        hooks.active = False; hooks.group = None; hooks.proj = None; hooks.raw = None; hooks.pidx = None; hooks.pd = None
    if pout is not None: assert not np.isnan(pout).any(), "a pair was not computed"
    return out, cnt, raws, pout


def fold_mean_diff(sums, counts, a, b):
    """sums/counts keyed by (type, fold): fold means of type a minus type b, [2, L, W, D] float32 on the CPU."""
    import torch
    return torch.stack([sums[(a, f)].cpu() / counts[(a, f)] - sums[(b, f)].cpu() / counts[(b, f)] for f in (0, 1)])


def ref_norms(refs, K):
    """Squared norm of each reference over the template part and the answer part of the window: [R, 2]."""
    r = np.asarray(refs, dtype=np.float64)
    return np.stack([(r[:, :, :K] ** 2).sum((1, 2, 3)), (r[:, :, K:] ** 2).sum((1, 2, 3))], 1)


def item_drive(proj_a, proj_b, ref_of_item, norms, part='window'):
    """Per item drive of the pair (a, b): proj_* [n, L, R, 2] for the same items in the same order; ref_of_item [n] the
    reference fold used for each item (the other fold); norms [R, 2]. part: 'window', 'template' or 'answer'.
    Returns (drive [n], per-layer contribution [n, L])."""
    d = proj_a - proj_b                                                             # [n, L, R, 2]
    rr = np.asarray(ref_of_item, dtype=int); n = d.shape[0]
    sel = d[np.arange(n), :, rr, :]                                                 # [n, L, 2]
    if part == 'template': num, den = sel[:, :, 0], norms[rr, 0]
    elif part == 'answer': num, den = sel[:, :, 1], norms[rr, 1]
    else: num, den = sel.sum(-1), norms[rr].sum(-1)
    per_layer = num / den[:, None]
    return per_layer.sum(1), per_layer


def cluster_boot(v, clusters, n_boot=5000, seed=0):
    """Mean of v with a percentile bootstrap that resamples clusters (host documents): (mean, lo, hi)."""
    v = np.asarray(v, dtype=float); cl = np.asarray(clusters); u, inv = np.unique(cl, return_inverse=True)
    sums = np.bincount(inv, weights=v, minlength=len(u)); cnts = np.bincount(inv, minlength=len(u)).astype(float)
    rng = np.random.default_rng(seed); k = len(u)
    pick = rng.integers(0, k, size=(n_boot, k))
    bs = sums[pick].sum(1) / cnts[pick].sum(1)
    return float(v.mean()), float(np.percentile(bs, 2.5)), float(np.percentile(bs, 97.5))


def pair_cosine(pd_, part='window'):
    """Per pair cosine between its two differences from pair results [P, L, 3, 2]: summed over layers (and parts) before
    dividing. Returns (cosine [P], per-layer cosine [P, L])."""
    sel = pd_.sum(-1) if part == 'window' else pd_[..., 0 if part == 'template' else 1]       # [P, L, 3]
    tot = sel.sum(1)
    cos = tot[:, 0] / np.sqrt(np.maximum(tot[:, 1] * tot[:, 2], 1e-30))
    per = sel[:, :, 0] / np.sqrt(np.maximum(sel[:, :, 1] * sel[:, :, 2], 1e-30))
    return cos, per


def spearman_perm(x, y, n_perm=20000, seed=0, strata=None):
    """Spearman's rho (average ranks for ties) and the one-sided permutation p-value for rho at least as large. With
    strata (one label per point), y is shuffled only within each stratum, so a shift shared by a whole group (a model
    family) cannot make the association on its own."""
    from scipy.stats import rankdata
    x = rankdata(np.asarray(x, dtype=float)); y = rankdata(np.asarray(y, dtype=float))
    rho = lambda a, b: float(np.corrcoef(a, b)[0, 1])
    r0 = rho(x, y); rng = np.random.default_rng(seed)
    groups = [np.arange(len(y))] if strata is None else [np.flatnonzero(np.asarray(strata) == g) for g in sorted(set(strata))]
    def shuffled():
        z = y.copy()
        for g in groups: z[g] = y[rng.permutation(g)]
        return z
    perm = np.array([rho(x, shuffled()) for _ in range(n_perm)])
    return r0, float((1 + (perm >= r0 - 1e-12).sum()) / (n_perm + 1))

## Stage A. The conditions, what each achieves and costs (from the repository), the items, the statements and the marks

In [ ]:
import importlib, hashlib, collections, numpy as np, json, re, pandas as pd
import task_state, head_patching, shared_drive, adaptive_attack, final_scoring, targets, model_defense, llama_h2h, fidelity_exact
for m_ in (task_state, head_patching, shared_drive, adaptive_attack, final_scoring, targets, model_defense, llama_h2h, fidelity_exact): importlib.reload(m_)
from task_state import suffix_ids
from head_patching import HeadHooks, capture_heads
from shared_drive import DriveHooks, read, encode_convs, ref_norms, item_drive, cluster_boot, pair_cosine, spearman_perm
from adaptive_attack import host_of, build_variant
from final_scoring import CATEGORIES
from targets import load_lm, TASK_SYSTEM, TASK_USER
from model_defense import TASK_USER_WITH
from fidelity_exact import split_sentences, _clean_sentence_ok, norm
H = lambda s: hashlib.sha1((s or '').encode()).hexdigest()
def jload(f): return json.load(open(f)) if os.path.exists(f) else {}
def jsave(obj, f): json.dump(obj, open(f + '.tmp', 'w')); os.replace(f + '.tmp', f)
pd.set_option('display.width', 260); pd.set_option('display.max_columns', 40); pd.set_option('display.max_colwidth', 90)
OUT, REP, FIG = 'data/shared_drive', 'reports/shared_drive', 'figures/shared_drive'
for d_ in (OUT, REP, FIG): os.makedirs(d_, exist_ok=True)

# ---- marks and settings, fixed before any result ----
SEED, BS, NANS, NBOOT, NPERM = 59, 16, 16, 5000, 20000
CRIT59 = dict(v_min=0.05, v_seeds=5, v_items=105, h1_rho=0.50, h1_p=0.05, h1_n=16, h2_rho=0.50, h2_p=0.05, h2_n=15, h3_hi=0.0)
print('marks:', CRIT59)

# ---- the 19 conditions: weights, prompt format, and where their attack rate, user measure and deletion gap are in the repository ----
QWEN3B, QWEN7B, SEC = 'Qwen/Qwen2.5-3B-Instruct', 'Qwen/Qwen2.5-7B-Instruct', llama_h2h.SECALIGN
FT, CS, CU, AA, U7, LU = 'reports/paper_fixed_all_models.csv', 'reports/controls_security.csv', 'reports/controls_utility.csv', 'reports/ablation_anchor_scale.csv', 'reports/fidelity_v5_7b_utility.csv', 'reports/llama_utility.csv'
CF, F7, LF = 'reports/controls_fidelity.csv', 'reports/fidelity_v5_7b_fidelity.csv', 'reports/llama_fidelity.csv'
def C_(tag, fam, adapter, label, asr, uf, dg, mode='plain', tokf=None): return dict(tag=tag, fam=fam, adapter=adapter, label=label, asr=asr, uf=uf, dg=dg, mode=mode, tokf=tokf)
CONDS = [
    C_('q3_undef',   'q3', None, 'Qwen 3B undefended', (FT, 'Qwen 3B undefended'), (CU, 'Qwen 3B undefended'), (CF, 'Qwen 3B undefended')),
    C_('q3_v1',      'q3', 'data/distill/lora_3b', 'Qwen 3B twin distillation v1', (FT, 'twin_distilled_v1'), (AA, 'twins, v1 scale (1,500 attacked + 1,500 twins)'), None),
    C_('q3_notwin1', 'q3', 'data/distill_notwin/lora_3b', 'Qwen 3B distillation without twins, v1 scale', (FT, 'no_twin'), (AA, 'no twin, v1 scale (1,500 attacked)'), None),
    C_('q3_notwin2', 'q3', 'data/distill_notwin2/lora_3b', 'Qwen 3B distillation without twins, v2 scale', (FT, 'no_twin_v2'), (CU, 'no twins (v2 scale)'), (CF, 'no twins (v2 scale)')),
    C_('q3_v2',      'q3', 'data/distill2/lora_3b', 'Qwen 3B twin distillation v2', (FT, 'Qwen 3B twin distillation v2'), (CU, 'v2 (twin distillation)'), (CF, 'v2 (twin distillation)')),
    C_('q3_v3',      'q3', 'data/distill3/lora_3b', 'Qwen 3B twin distillation v3', (FT, 'twin_distilled_v3'), None, None),
    C_('q3_v4',      'q3', 'data/distill4/lora_3b', 'Qwen 3B full recipe (v4)', (FT, 'Qwen 3B full recipe (final)'), (CU, 'v4 = v2 + literal phase (final)'), (CF, 'v4 = v2 + literal phase (final)')),
    C_('q3_ntlit',   'q3', 'data/controls/notwin_literal/lora_3b', 'Qwen 3B no twins + literal phase', (CS, 'no twins + literal phase'), (CU, 'no twins + literal phase'), (CF, 'no twins + literal phase')),
    C_('q3_replay',  'q3', 'data/controls/replay_only/lora_3b', 'Qwen 3B v2 + replay only', (CS, 'v2 + replay only (equal budget)'), (CU, 'v2 + replay only (equal budget)'), (CF, 'v2 + replay only (equal budget)')),
    C_('q3_stlit',   'q3', 'data/controls/statement_literal/lora_3b', 'Qwen 3B v2 + statement-only literal phase', (CS, 'v2 + literal phase, statements only'), (CU, 'v2 + literal phase, statements only'), (CF, 'v2 + literal phase, statements only')),
    C_('q7_undef',   'q7', None, 'Qwen 7B undefended', (FT, 'Qwen 7B undefended'), (U7, 'undefended 7B'), (F7, 'undefended_7b')),
    C_('q7_notwin',  'q7', 'data/distill7b/lora_7b', 'Qwen 7B distillation without twins', (FT, 'twin_distilled_7b'), (U7, 'distilled 7B (no twins)'), (F7, 'twin_distilled_7b')),
    C_('q7_v4',      'q7', 'data/distill7b_v4/lora_7b', 'Qwen 7B twins + literal (v4)', (FT, 'fidelity_v4_7b'), (U7, 'fidelity_v4_7b (twins + literal)'), (F7, 'fidelity_v4_7b')),
    C_('q7_v5',      'q7', 'data/distill7b_v5/lora_7b', 'Qwen 7B full recipe (v5)', (FT, 'Qwen 7B full recipe (final)'), (U7, 'fidelity_v5_7b (rebalanced toward the defense)'), (F7, 'fidelity_v5_7b')),
    C_('l_undef',    'll', None, 'Llama 8B undefended', (FT, 'Llama 8B undefended'), (LU, 'undefended Llama-3.1-8B'), (LF, 'undefended Llama-3.1-8B')),
    C_('l_p1',       'll', 'data/llama/phase1/lora', 'Llama 8B ours, phase 1', (FT, 'llama_twin_p1'), (LU, 'ours, phase 1 (twin distillation)'), (LF, 'ours, phase 1 (twin distillation)')),
    C_('l_full',     'll', 'data/llama/phase2/lora', 'Llama 8B ours, full recipe', (FT, 'Llama 8B ours, full recipe'), (LU, 'ours, full recipe'), (LF, 'ours, full recipe')),
    C_('l_sec',      'll', SEC, 'Meta-SecAlign-8B, input role (as specified)', (FT, 'Meta-SecAlign-8B (as specified)'), (LU, 'Meta-SecAlign-8B (as specified)'), (LF, 'Meta-SecAlign-8B (as specified)'), mode='input_role', tokf=SEC),
    C_('l_sec_user', 'll', SEC, 'Meta-SecAlign-8B, document in the user turn', (FT, 'Meta-SecAlign-8B (data in the user turn)'), (LU, 'Meta-SecAlign-8B (data in the user turn)'), (LF, 'Meta-SecAlign-8B (data in the user turn)'), mode='plain', tokf=SEC)]
CD = {c['tag']: c for c in CONDS}; assert len(CD) == 19
FAM = {'q3': dict(base=QWEN3B, label='Qwen 3B', llama=False), 'q7': dict(base=QWEN7B, label='Qwen 7B', llama=False), 'll': dict(base=llama_h2h.BASE, label='Llama 8B', llama=True)}
BASE_OF = {f: next(c['tag'] for c in CONDS if c['fam'] == f and c['adapter'] is None) for f in FAM}
for c in CONDS:
    if c['adapter'] and c['adapter'] != SEC: assert os.path.exists(os.path.join(c['adapter'], 'adapter_config.json')), f"missing adapter {c['adapter']}"

# ---- what each condition achieves and costs, read from the repository's tables ----
def num(s):
    m_ = re.match(r'\s*([+-]?\d*\.?\d+)', str(s)); return float(m_.group(1)) if m_ else np.nan
def cell(path, label, col):
    T_ = pd.read_csv(path); r_ = T_[T_.iloc[:, 0] == label]; assert len(r_) == 1, f'{path}: {len(r_)} rows named {label!r}'; return num(r_[col].iloc[0])
def delgap(path, label):
    T_ = pd.read_csv(path); r_ = T_[T_.iloc[:, 0] == label]
    assert len(r_) == 3 and set(r_.task) == {'repeat', 'number', 'quote_last'}, f'{path}: rows for {label!r}'
    return float(np.mean([num(x) for x in r_.deletion_gap]))
COST = {c['tag']: dict(asr=cell(c['asr'][0], c['asr'][1], 'fixed_13') if c['asr'] else np.nan,
                       uf=cell(c['uf'][0], c['uf'][1], 'user_followed_150') if c['uf'] else np.nan,
                       dg=delgap(*c['dg']) if c['dg'] else np.nan) for c in CONDS}
assert cell(AA, 'undefended', 'user_followed_150') == COST['q3_undef']['uf'], 'the two 3B user-following tables disagree on the undefended model'
assert all(cell(CS, a_, 'fixed_13') == cell(FT, b_, 'fixed_13') for a_, b_ in (('Qwen 3B undefended', 'Qwen 3B undefended'), ('v2 (twin distillation)', 'Qwen 3B twin distillation v2'), ('v4 = v2 + literal phase (final)', 'Qwen 3B full recipe (final)'), ('no twins (v2 scale)', 'no_twin_v2'))), 'the controls\' attack table and the paper\'s disagree on the models they share'
for c in CONDS:
    b_ = BASE_OF[c['fam']]; COST[c['tag']].update(d_asr=COST[c['tag']]['asr'] - COST[b_]['asr'], d_uf=COST[c['tag']]['uf'] - COST[b_]['uf'])
CT = pd.DataFrame([dict(condition=c['tag'], label=c['label'], **{k: round(v, 3) for k, v in COST[c['tag']].items()}) for c in CONDS])
print('\n=== what each condition achieves and costs (from the repository, the paper\'s rule; d_ = minus its own undefended model) ==='); print(CT.to_string(index=False))
H1_SET = [c['tag'] for c in CONDS if c['adapter'] and COST[c['tag']]['asr'] == COST[c['tag']]['asr']]
H2_SET = [c['tag'] for c in CONDS if c['adapter'] and COST[c['tag']]['uf'] == COST[c['tag']]['uf']]
assert len(H1_SET) == CRIT59['h1_n'] and len(H2_SET) == CRIT59['h2_n'], (len(H1_SET), len(H2_SET))

# ---- the 210 injections, their clean hosts, and an ordinary sentence for each ----
SUB = pd.DataFrame(json.load(open('data/phase5v3/subset_full.json'))); n = len(SUB); assert n == 210 and set(SUB.meta) == set(CATEGORIES)
SPLIT = [host_of(SUB.text.iloc[i], SUB.instruction.iloc[i]) for i in range(n)]
assert all(a_ is not None for _, a_ in SPLIT), 'an injection is not verbatim in its document'
CLEAN_DOC = [(b_[:-1] if b_.endswith('\n') else b_) + a_ for b_, a_ in SPLIT]
assert open('data/task_state/qwen3b_undefended/clean_docs.sha1').read().strip() == H('\x00'.join(CLEAN_DOC)), 'the clean twins differ from notebook 51\'s'
INS = [str(SUB.instruction.iloc[i]) for i in range(n)]
assert all(build_variant(SPLIT[i][0], SPLIT[i][1], INS[i]) == SUB.text.iloc[i] for i in range(n)), 'rebuilding an attacked document from its host does not give it back'
HOSTS = sorted(set(CLEAN_DOC)); HID = {d: k for k, d in enumerate(HOSTS)}; HOST = np.array([HID[d] for d in CLEAN_DOC])
_IMP = re.compile(r"^(please|ignore|write|translate|tell|say|give|make|list|summari[sz]e|provide|include|add|use|do|don't|let|answer|respond|reply|describe|explain|create|send|click|visit|check|find|call|contact|read|follow|replace|convert|encode|decode|reverse|rewrite|output|print|start|forget|disregard|remember|note|keep|stop|try|go|see|share|show|change|put|insert|append|repeat|copy|type|enter|return|consider|imagine)\b", re.I)
def ordinary(s): return _clean_sentence_ok(s, 4, 40) and s.endswith('.') and '\n' not in s and not _IMP.match(s) and not re.search(r"\b(please|you|your)\b", s, re.I)
# the ordinary sentences come from SQuAD validation paragraphs (Wikipedia), the source of notebook 37's statements; the BIPIA hosts hold too few clean prose sentences
try:
    from datasets import load_dataset
    SRC_TXT = sorted(set(load_dataset('rajpurkar/squad', split='validation')['context'])); SRC_NAME = 'SQuAD validation paragraphs'
except Exception as e_:
    print('SQuAD could not be loaded (' + type(e_).__name__ + '); falling back to notebook 37\'s saved statements')
    SRC_TXT = sorted({it['statement'] for it in json.load(open('data/fidelity_exact/items.json'))}); SRC_NAME = "notebook 37's statements"
POOL = sorted({s for d in SRC_TXT for s in split_sentences(d) if ordinary(s)})
print(f'\nordinary sentences available: {len(POOL)} from {SRC_NAME}')
assert len(POOL) >= 200, 'too few ordinary sentences to build the statement control'
wc = lambda s: len(s.split())
POOL_WC = np.array([wc(s) for s in POOL]); POOL_N = [norm(s) for s in POOL]; STMT = []
for i in range(n):
    gap = np.abs(POOL_WC - wc(INS[i])); order_ = np.lexsort((np.random.default_rng(SEED + i).random(len(POOL)), gap))      # nearest in length first, ties in a seeded random order
    cand = [POOL[j] for j in order_[:40] if POOL_N[j] not in norm(CLEAN_DOC[i])][:10]
    assert cand, f'no statement for item {i}'
    STMT.append(cand[int(np.random.default_rng(SEED + i).integers(len(cand)))])
ST = pd.DataFrame(dict(item=range(n), category=SUB.meta, instruction=INS, statement=STMT, words_instruction=[wc(x) for x in INS], words_statement=[wc(x) for x in STMT]))
ST.to_csv(f'{REP}/statements.csv', index=False)
print(f'words, median: instruction {int(ST.words_instruction.median())}, statement {int(ST.words_statement.median())} | distinct statements used {ST.statement.nunique()} | most uses of one statement {ST.statement.value_counts().iloc[0]}')
print(ST.sample(4, random_state=SEED)[['category', 'instruction', 'statement']].to_string(index=False))

# ---- two halves of the items by host document (an item is read against the direction built on the other half) ----
perm_ = np.random.default_rng(SEED).permutation(len(HOSTS)); FOLD_OF_HOST = np.zeros(len(HOSTS), int); FOLD_OF_HOST[perm_[1::2]] = 1
FOLD = FOLD_OF_HOST[HOST]
print('items per half:', int((FOLD == 0).sum()), int((FOLD == 1).sum())); assert min((FOLD == 0).sum(), (FOLD == 1).sum()) >= 80
# ---- each item is read against a user direction built from the other half of the hosts and from the other six kinds of task only, so it shares neither its host nor its instruction's wording ----
CATL = sorted(set(SUB.meta)); CAT = np.array([CATL.index(c_) for c_ in SUB.meta]); NREF = 2 * len(CATL)
REF_OF = CAT * 2 + (1 - FOLD)                                                     # reference index: (left-out kind of task, half it was built on)
for ci in range(len(CATL)):
    for f in (0, 1): assert ((CAT != ci) & (FOLD == f)).sum() >= 60, 'too few items to build a reference'

# ---- the five prompts per item, read together (one item's five prompts always share a forward pass) ----
# each prompt is kept as (task part, document): the plain format joins them with a blank line, exactly as TASK_USER and TASK_USER_WITH do; the input-role
# format puts the task part in the user role and the document in the input role, as llama_h2h.messages_for does, without relying on a blank-line split
T_HEAD, TW_HEAD = TASK_USER.split('\n\n{doc}')[0], TASK_USER_WITH.split('\n\n{doc}')[0]
assert TASK_USER == T_HEAD + '\n\n{doc}' and TASK_USER_WITH == TW_HEAD + '\n\n{doc}' and '{doc}' not in T_HEAD + TW_HEAD
TYPES = ('P', 'D_ins', 'D_stmt', 'U_ins', 'U_stmt')
PART = {'P': [(T_HEAD, CLEAN_DOC[i]) for i in range(n)],
        'D_ins': [(T_HEAD, SUB.text.iloc[i]) for i in range(n)],
        'D_stmt': [(T_HEAD, build_variant(SPLIT[i][0], SPLIT[i][1], STMT[i])) for i in range(n)],
        'U_ins': [(TW_HEAD.format(instr=INS[i]), CLEAN_DOC[i]) for i in range(n)],
        'U_stmt': [(TW_HEAD.format(instr=STMT[i]), CLEAN_DOC[i]) for i in range(n)]}
PROMPT = {t: [h_ + '\n\n' + d_ for h_, d_ in PART[t]] for t in TYPES}
assert PROMPT['P'][0] == TASK_USER.format(doc=CLEAN_DOC[0]) and PROMPT['U_ins'][0] == TASK_USER_WITH.format(instr=INS[0], doc=CLEAN_DOC[0])
def conv(t, i, mode):
    h_, d_ = PART[t][i]; sys_ = [{'role': 'system', 'content': TASK_SYSTEM}]
    if mode == 'plain': return sys_ + [{'role': 'user', 'content': h_ + '\n\n' + d_}]
    if d_.startswith('Input:\n'): d_ = d_[len('Input:\n'):]
    return sys_ + [{'role': 'user', 'content': h_}, {'role': 'input', 'content': llama_h2h.recursive_filter(d_)}]
NBL = sum('\n\n' in h_ for t in TYPES for h_, _ in PART[t])
print(f'task parts holding a blank line (an instruction of several paragraphs): {NBL}; the input-role prompts are built from the parts, so these keep the whole instruction in the user role')
for mode in ('plain', 'input_role'):                                                  # where the project's own helper splits correctly, the prompts are identical to it
    same = [conv(t, i, mode) == llama_h2h.messages_for(PROMPT[t][i], mode, TASK_SYSTEM) for t in TYPES for i in range(n) if '\n\n' not in PART[t][i][0]]
    assert all(same), f'{mode}: {len(same) - sum(same)} prompts differ from llama_h2h.messages_for'
ROWS = [(t, i) for t in TYPES for i in range(n)]; RIX = {ri: r for r, ri in enumerate(ROWS)}; GROUP = [(t, int(FOLD[i])) for t, i in ROWS]
ROWS_OF = {t: np.array([RIX[(t, i)] for i in range(n)]) for t in TYPES}
PAIRS = [(RIX[('D_ins', i)], RIX[('U_ins', i)], RIX[('P', i)], RIX[('P', i)]) for i in range(n)] + [(RIX[('D_stmt', i)], RIX[('U_stmt', i)], RIX[('P', i)], RIX[('P', i)]) for i in range(n)]
ITEM_ORDER = sorted(range(n), key=lambda i: (len(CLEAN_DOC[i]), i)); BI = 4
BATCH5 = [[RIX[(t, i)] for i in ITEM_ORDER[s_: s_ + BI] for t in TYPES] for s_ in range(0, n, BI)]
V_ITEMS = ITEM_ORDER[::2][:CRIT59['v_items']]; assert len(V_ITEMS) == CRIT59['v_items']                     # the items the random-weight readings use
PH = H(json.dumps([PROMPT[t][i] for t, i in ROWS]))[:12]
def convs_of(mode): return [conv(t, i, mode) for t, i in ROWS]
print(f'prompts: {len(ROWS)} per condition in {len(BATCH5)} passes of {BI} items (hash {PH})')

## Stage B. Loading, the shared answer tokens, and the reader's self-test on the 3B base model

In [ ]:
from peft import PeftModel
M = dict(model=None, tok=None, tag=None, fam=None)
def free_model():
    M.update(model=None, tok=None, tag=None, fam=None); gc.collect(); torch.cuda.empty_cache()
def load_cond(tag):
    """The weights and tokenizer of a condition exactly as notebooks 30 to 45 loaded them."""
    c = CD[tag]; f = FAM[c['fam']]
    if M['tag'] == tag or (M['model'] is not None and M['tag'] in CD and M['fam'] == c['fam'] and CD[M['tag']]['adapter'] == c['adapter'] and CD[M['tag']]['tokf'] == c['tokf']):
        M['tag'] = tag; return M['model'], M['tok']                                      # the same weights in another prompt format (Meta-SecAlign)
    free_model()
    if f['llama']: m_, t_ = llama_h2h.load_llama(adapter=c['adapter'], tokenizer_from=c['tokf'])
    else:
        m_, t_ = load_lm(f['base'], four_bit=True)
        if c['adapter']: m_ = PeftModel.from_pretrained(m_, c['adapter']).eval()
    M.update(model=m_, tok=t_, tag=tag, fam=c['fam']); print(f'loaded {c["label"]}' + (f' ({c["adapter"]})' if c['adapter'] else '')); return m_, t_
def load_random(fam, seed):
    """The family's architecture with random weights (seeded), in bf16: one draw of the null for sharedness. Same tokenizer, same prompts, same answer tokens."""
    from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer
    free_model(); cfg = AutoConfig.from_pretrained(FAM[fam]['base']); torch.manual_seed(1000 * SEED + seed)
    with torch.device('cuda' if torch.cuda.is_available() else 'cpu'): m_ = AutoModelForCausalLM.from_config(cfg, torch_dtype=torch.bfloat16).eval()
    t_ = AutoTokenizer.from_pretrained(FAM[fam]['base'])
    if t_.pad_token is None: t_.pad_token = t_.eos_token
    M.update(model=m_, tok=t_, tag=f'{fam}_random{seed}', fam=fam); return m_, t_
def answers_for(fam):
    """16 answer tokens per item: the base model's greedy start of an answer to D_stmt, generated once and shared by
    every prompt of the item and every condition of the family."""
    fp = f'{OUT}/answers_{fam}.json'; A_ = jload(fp)
    if A_.get('ph') == PH and len(A_.get('ids', [])) == n and all(len(a_) == NANS for a_ in A_['ids']): return A_
    m_, t_ = load_cond(BASE_OF[fam]); sfx = suffix_ids(t_, TASK_SYSTEM)
    cv = [conv('D_stmt', i, 'plain') for i in range(n)]; ids_ = [None] * n
    for s in range(0, n, BS):
        ch = list(range(s, min(n, s + BS))); ii, am = encode_convs(t_, [cv[i] for i in ch], sfx); dev = m_.get_input_embeddings().weight.device
        with torch.no_grad():
            g = m_.generate(input_ids=ii.to(dev), attention_mask=am.to(dev), max_new_tokens=NANS, min_new_tokens=NANS, do_sample=False, temperature=None, top_p=None, pad_token_id=t_.pad_token_id)
        for j, i in enumerate(ch): ids_[i] = [int(x) for x in g[j, ii.shape[1]:]]
    assert all(len(a_) == NANS for a_ in ids_), 'an answer start is shorter than 16 tokens'
    A_ = dict(ph=PH, vocab=len(t_), ids=ids_, text=[t_.decode(a_, skip_special_tokens=True) for a_ in ids_]); jsave(A_, fp)
    print(f'{FAM[fam]["label"]}: answer starts generated, e.g. {A_["text"][0]!r}'); return A_

# ---- the self-test, on the 3B base model ----
m, tok = load_cond('q3_undef'); SFX3 = suffix_ids(tok, TASK_SYSTEM); K = len(SFX3); W = K + NANS
ANS = {'q3': answers_for('q3')}
def self_test():
    its = ITEM_ORDER[:2]; rows = [RIX[(t, i)] for i in its for t in TYPES]; loc = {r: j for j, r in enumerate(rows)}
    cv = convs_of('plain'); sc = [cv[r] for r in rows]; sg = [GROUP[r] for r in rows]; sa = [ANS['q3']['ids'][ROWS[r][1]] for r in rows]
    sp = [tuple(loc[x] for x in PAIRS[q]) for q in (its[0], its[1], n + its[0], n + its[1])]
    hk = DriveHooks(m, W, K); Rr = torch.randn(2, hk.L, W, hk.D, generator=torch.Generator().manual_seed(SEED)); hk.set_refs(Rr)
    pr, cnt, raw, pdo = read(m, hk, tok, sc, sg, SFX3, ans_ids=sa, keep_raw=True, batches=[list(range(len(rows)))], pairs=sp)
    man = np.stack([np.stack([np.stack([np.einsum('kd,rkd->r', raw[j][l][:K].numpy(), Rr[:, l, :K].numpy()), np.einsum('kd,rkd->r', raw[j][l][K:].numpy(), Rr[:, l, K:].numpy())], -1) for l in range(hk.L)]) for j in range(len(rows))])
    e_proj = float(np.abs(man - pr).max() / np.abs(man).max())
    e_pair = 0.0
    for q, (a_, b_, c_, d_) in enumerate(sp):
        A_ = (raw[a_] - raw[c_]).flatten(); B__ = (raw[b_] - raw[d_]).flatten()
        e_pair = max(e_pair, abs(float(pair_cosine(pdo[q:q + 1])[0][0]) - float(A_ @ B__ / (A_.norm() * B__.norm()))))
    _, _, raw1, _ = read(m, hk, tok, sc[2:3], sg[2:3], SFX3, ans_ids=sa[2:3], batch=1, keep_raw=True)
    cosf = lambda a_, b_: float((a_.flatten() * b_.flatten()).sum() / (a_.norm() * b_.norm()))
    c_alone = cosf(raw1[0], raw[2])
    hh = HeadHooks(m, K); cap = capture_heads(m, hh, tok, [PROMPT[ROWS[r][0]][ROWS[r][1]] for r in rows], TASK_SYSTEM, SFX3); hh.close()
    c_cap = min(cosf(cap[j].float(), raw[j][:, :K]) for j in range(len(rows)))
    hk.close()
    print(f'self-test on {CD["q3_undef"]["label"]}: layers {hk.L}, heads {hk.nh} of size {hk.dh}, window {W} ({K} template positions + {NANS} answer tokens); '
          f'projections against a direct computation {e_proj:.1e}; item-level cosines against a direct computation {e_pair:.1e}; one row read alone against the same row in a batch of ten, cosine {c_alone:.6f}; '
          f'template part against the reading of notebooks 56 to 58 (the prompt pass of generate), lowest cosine {c_cap:.6f}')
    assert e_proj < 1e-3 and e_pair < 1e-4 and c_alone > 0.999 and c_cap > 0.999, 'the reader does not behave on this model: stop'
self_test()
del m                                                                                # the model stays in M for Stage C; this name must not pin it on the GPU

# ---- the checks that need only tokenizers, done now rather than hours into Stage C ----
from transformers import AutoTokenizer
for f in ('q7', 'll'):
    if FAM[f]['llama']: llama_h2h.hf_login()                                         # Llama and Meta-SecAlign are gated: the Colab secret HF_TOKEN, or a hidden box here
    tf_ = AutoTokenizer.from_pretrained(FAM[f]['base']); assert len(suffix_ids(tf_, TASK_SYSTEM)) == K, f'{f}: the template suffix is not {K} tokens'
st_ = AutoTokenizer.from_pretrained(SEC); llama_h2h.check_input_role_template(st_)
if st_.pad_token is None: st_.pad_token = st_.eos_token
encode_convs(st_, [conv('U_ins', i, 'input_role') for i in range(3)], suffix_ids(tf_, TASK_SYSTEM))   # fails here if the input role ends differently
print('tokenizer checks passed: Qwen 7B and Llama have the same', K, 'template positions; Meta-SecAlign\'s input role ends with the base template suffix')
del tf_, st_

## Stage C. Every condition read: one forward pass per prompt, cached per condition

In [ ]:
def ref_for(fam, sfx):
    """The family's user direction: in the undefended base model, U_ins minus U_stmt, one fold mean per half of the items."""
    A_ = ANS[fam]; key = f'{PH}|{H(json.dumps(A_["ids"]))[:12]}|{W}'; fp = f'{OUT}/ref_{fam}.pt'
    if os.path.exists(fp):
        d_ = torch.load(fp, weights_only=False)
        if d_.get('key') == key: return d_['u'], key
    m_, t_ = load_cond(BASE_OF[fam]); rows = [RIX[(t, i)] for i in ITEM_ORDER for t in ('U_ins', 'U_stmt')]; cv = convs_of('plain'); sub_c = [cv[r] for r in rows]
    hk = DriveHooks(m_, W, K)
    _, cnt, _, _ = read(m_, hk, t_, sub_c, [(ROWS[r][0], int(FOLD[ROWS[r][1]]), int(CAT[ROWS[r][1]])) for r in rows], sfx, ans_ids=[A_['ids'][ROWS[r][1]] for r in rows], batch=BS)
    S_ = {g: s_.cpu() for g, s_ in hk.sums.items()}; hk.close()
    def mean_excl(t, f, ci):
        ks = [(t, f, cj) for cj in range(len(CATL)) if cj != ci and (t, f, cj) in S_]
        return sum(S_[k_] for k_ in ks) / sum(cnt[k_] for k_ in ks)
    u = torch.stack([mean_excl('U_ins', f, ci) - mean_excl('U_stmt', f, ci) for ci in range(len(CATL)) for f in (0, 1)])     # [2 x kinds, L, W, D], index kind * 2 + half
    torch.save(dict(key=key, u=u), fp); print(f'{FAM[fam]["label"]}: {len(u)} user directions built ({len(rows)} prompts)'); return u, key
def run_random(fam, seed, rkey, sfx):
    """One random-weight draw of the family's architecture, read on the V items: the item-level cosines only."""
    tag = f'{fam}_random{seed}'; key = f'{rkey}|random|{SEED}|{seed}|{H(json.dumps(V_ITEMS))[:8]}'; fp = f'{OUT}/{tag}.pt'
    if os.path.exists(fp):
        d_ = torch.load(fp, weights_only=False)
        if d_.get('key') == key: return d_
    m_, t_ = load_random(fam, seed); assert suffix_ids(t_, TASK_SYSTEM) == sfx
    rows = [RIX[(t, i)] for i in V_ITEMS for t in TYPES]; loc = {r: j for j, r in enumerate(rows)}; cv = convs_of('plain')
    prs = [tuple(loc[x] for x in PAIRS[i]) for i in V_ITEMS] + [tuple(loc[x] for x in PAIRS[n + i]) for i in V_ITEMS]
    hk = DriveHooks(m_, W, K)
    _, _, _, pdo = read(m_, hk, t_, [cv[r] for r in rows], [GROUP[r] for r in rows], sfx, ans_ids=[ANS[fam]['ids'][ROWS[r][1]] for r in rows],
                        batches=[list(range(s_, min(len(rows), s_ + BI * len(TYPES)))) for s_ in range(0, len(rows), BI * len(TYPES))], pairs=prs)
    hk.close(); free_model(); d_ = dict(key=key, pairs=pdo); torch.save(d_, fp); print(f'  {FAM[fam]["label"]} with random weights, draw {seed}: read, {len(rows)} prompts'); return d_
def run_cond(tag, u, rkey, sfx):
    c = CD[tag]; key = f'{rkey}|{c["adapter"]}|{c["mode"]}|{c["tokf"]}'; fp = f'{OUT}/{tag}.pt'
    if os.path.exists(fp):
        d_ = torch.load(fp, weights_only=False)
        if d_.get('key') == key: return d_
    m_, t_ = load_cond(tag)
    assert suffix_ids(t_, TASK_SYSTEM) == sfx, f'{tag}: the template suffix differs from the base model\'s'
    assert [t_.decode(a_, skip_special_tokens=True) for a_ in ANS[c['fam']]['ids'][:20]] == ANS[c['fam']]['text'][:20], f'{tag}: this tokenizer reads the shared answer tokens differently from the base model\'s'
    cv = convs_of(c['mode']); hk = DriveHooks(m_, W, K); hk.set_refs(u)
    proj, cnt, _, pdo = read(m_, hk, t_, cv, GROUP, sfx, ans_ids=[ANS[c['fam']]['ids'][i] for _, i in ROWS], batches=BATCH5, pairs=PAIRS)
    d_ = dict(key=key, proj=proj, pairs=pdo, cnt=cnt, nh=hk.nh, L=hk.L)
    hk.close(); torch.save(d_, fp); print(f'  {c["label"]}: read, {len(ROWS)} prompts'); return d_

RES, U_REF, NORMS, SFX = {}, {}, {}, {}
for fam in ('q3', 'q7', 'll'):
    tags = [c['tag'] for c in CONDS if c['fam'] == fam]
    if fam not in ANS: ANS[fam] = answers_for(fam)
    t0_ = load_cond(BASE_OF[fam])[1] if not all(os.path.exists(f'{OUT}/{t}.pt') for t in tags) else None
    if t0_ is None:
        from transformers import AutoTokenizer
        t0_ = AutoTokenizer.from_pretrained(FAM[fam]['base'])
    SFX[fam] = suffix_ids(t0_, TASK_SYSTEM); assert len(SFX[fam]) == K, f'{fam}: {len(SFX[fam])} template positions, not {K}'
    U_REF[fam], rkey = ref_for(fam, SFX[fam]); NORMS[fam] = ref_norms(U_REF[fam].numpy(), K)
    print(f'{FAM[fam]["label"]}: squared size of the {len(NORMS[fam])} user directions, template part {NORMS[fam][:, 0].min():.3g} to {NORMS[fam][:, 0].max():.3g}, answer part {NORMS[fam][:, 1].min():.3g} to {NORMS[fam][:, 1].max():.3g}')
    for t in tags:
        RES[t] = run_cond(t, U_REF[fam], rkey, SFX[fam])
        if t != BASE_OF[fam]:
            d_ = np.abs(RES[t]['proj'] - RES[BASE_OF[fam]]['proj']).max(); s_ = np.abs(RES[BASE_OF[fam]]['proj']).max()
            assert d_ > 1e-3 * s_, f'{t}: its readings equal the base model\'s, so the adapter is not active'
    for sd_ in range(CRIT59['v_seeds']): RES[f'{fam}_random{sd_}'] = run_random(fam, sd_, rkey, SFX[fam])
free_model()
print(f'\nall 19 conditions and {3 * CRIT59["v_seeds"]} random-weight readings done')

## Stage D. The drives, the marks and what they decide

In [ ]:
import matplotlib.pyplot as plt
from reslog import log_result
fmt = lambda t3: f'{t3[0]:+.3f} [{t3[1]:+.3f}, {t3[2]:+.3f}]'
DR = {}
for t, fam_ in [(c['tag'], c['fam']) for c in CONDS]:
    P_ = RES[t]['proj']; PD_ = RES[t]['pairs']; nr = NORMS[fam_]; d_ = {}
    for part in ('window', 'template', 'answer'):
        ci_, pli = pair_cosine(PD_[:n], part); cs_, pls = pair_cosine(PD_[n:], part)
        d_[f'cos_ins_{part}'], d_[f'cos_stmt_{part}'], d_[f'shared_{part}'] = ci_, cs_, ci_ - cs_
        d_[f'inj_{part}'], pl_i = item_drive(P_[ROWS_OF['D_ins']], P_[ROWS_OF['D_stmt']], REF_OF, nr, part)
        d_[f'usr_{part}'], pl_u = item_drive(P_[ROWS_OF['U_ins']], P_[ROWS_OF['U_stmt']], REF_OF, nr, part)
        if part == 'window': d_['layers_shared'], d_['layers_inj'], d_['layers_usr'] = (pli - pls).mean(0), pl_i.mean(0), pl_u.mean(0)
    DR[t] = d_
rows, B_ = [], {}
for c in CONDS:
    t = c['tag']; b_ = BASE_OF[c['fam']]; d_ = DR[t]; bt = lambda k: cluster_boot(d_[k], HOST, NBOOT, SEED)
    ch = lambda k: cluster_boot(d_[k] - DR[b_][k], HOST, NBOOT, SEED) if t != b_ else (0.0, 0.0, 0.0)
    B_[t] = dict(shared=bt('shared_window'), usr=bt('usr_window'), inj=bt('inj_window'), dshared=ch('shared_window'), dusr=ch('usr_window'), dinj=ch('inj_window'))
    rows.append(dict(condition=t, label=c['label'], sharedness=fmt(B_[t]['shared']), cos_instruction=round(float(d_['cos_ins_window'].mean()), 3), cos_statement=round(float(d_['cos_stmt_window'].mean()), 3),
                     sharedness_template=round(float(d_['shared_template'].mean()), 3), sharedness_answer=round(float(d_['shared_answer'].mean()), 3),
                     user_drive=fmt(B_[t]['usr']), injected_drive=fmt(B_[t]['inj']),
                     change_sharedness=fmt(B_[t]['dshared']) if t != b_ else '', change_user_drive=fmt(B_[t]['dusr']) if t != b_ else '', change_injected_drive=fmt(B_[t]['dinj']) if t != b_ else '',
                     attack_13=COST[t]['asr'], change_attack=round(COST[t]['d_asr'], 3), user_followed=COST[t]['uf'], change_user_followed=round(COST[t]['d_uf'], 3), deletion_gap=round(COST[t]['dg'], 3)))
# ---- the random-weight readings on the V items: one value of sharedness per draw ----
VI = np.array(V_ITEMS); RNDV, RNDL = {}, {}
for f in FAM:
    vals, lays = [], []
    for sd_ in range(CRIT59['v_seeds']):
        pd_ = RES[f'{f}_random{sd_}']['pairs']; m_ = len(VI)
        ci_, li_ = pair_cosine(pd_[:m_]); cs_, ls_ = pair_cosine(pd_[m_:]); vals.append((float(ci_.mean()), float(cs_.mean()), float((ci_ - cs_).mean()))); lays.append((li_ - ls_).mean(0))
        rows.append(dict(condition=f'{f}_random{sd_}', label=f'{FAM[f]["label"]} architecture, random weights, draw {sd_} ({m_} items)', sharedness=f'{vals[-1][2]:+.3f}', cos_instruction=round(vals[-1][0], 3), cos_statement=round(vals[-1][1], 3)))
    RNDV[f] = vals; RNDL[f] = np.mean(lays, 0)
DRT = pd.DataFrame(rows); DRT.to_csv(f'{REP}/drive.csv', index=False)
LY = pd.DataFrame([dict(model=CD[BASE_OF[f]]['label'], layer=l, depth=round(l / (len(DR[BASE_OF[f]]['layers_shared']) - 1), 3), sharedness=round(float(DR[BASE_OF[f]]['layers_shared'][l]), 4),
                        sharedness_random_weights=round(float(RNDL[f][l]), 4), injected_drive=round(float(DR[BASE_OF[f]]['layers_inj'][l]), 4), user_drive=round(float(DR[BASE_OF[f]]['layers_usr'][l]), 4))
                   for f in FAM for l in range(len(DR[BASE_OF[f]]['layers_shared']))]); LY.to_csv(f'{REP}/layers.csv', index=False)
print('=== per condition: sharedness (the same sentence in the document and in the user turn, instruction minus ordinary sentence), the plain cosines behind it, and the drives along the user direction ===')
print(DRT.to_string(index=False))

# ---- the marks ----
BV = {f: cluster_boot(DR[BASE_OF[f]]['shared_window'][VI], HOST[VI], NBOOT, SEED) for f in FAM}                   # the undefended model on the same items
V_ = {f: BV[f][0] - max(v[2] for v in RNDV[f]) for f in FAM}; vok = lambda v: v >= CRIT59['v_min']; V = all(vok(v) for v in V_.values())
x1 = [B_[t]['dshared'][0] for t in H1_SET]; y1 = [COST[t]['d_asr'] for t in H1_SET]; r1, p1 = spearman_perm(x1, y1, NPERM, SEED, strata=[CD[t]['fam'] for t in H1_SET])
x2 = [B_[t]['dusr'][0] for t in H2_SET]; y2 = [COST[t]['d_uf'] for t in H2_SET]; r2, p2 = spearman_perm(x2, y2, NPERM, SEED, strata=[CD[t]['fam'] for t in H2_SET])
H1 = r1 >= CRIT59['h1_rho'] and p1 < CRIT59['h1_p']; H2 = r2 >= CRIT59['h2_rho'] and p2 < CRIT59['h2_p']
h3 = cluster_boot(DR['l_sec']['shared_window'] - DR['l_sec_user']['shared_window'], HOST, NBOOT, SEED); H3 = h3[2] < CRIT59['h3_hi']
checks = [
    ('V', f'sharedness of each undefended model on the {len(VI)} V items minus the highest of its architecture\'s {CRIT59["v_seeds"]} random-weight draws', '; '.join(f'{FAM[f]["label"]} {v:+.3f}' for f, v in V_.items()),
     f'>= {CRIT59["v_min"]:.2f} on all three', V),
    ('V (reported)', 'sharedness on the V items: undefended model [interval]; random-weight draws', '; '.join(f'{FAM[f]["label"]} {fmt(BV[f])}; ' + ', '.join(f'{v[2]:+.3f}' for v in RNDV[f]) for f in FAM), 'reported', True),
    ('H1', f'Spearman rho, change in sharedness against change in attack success (paper rule, 13 fixed tries), {len(H1_SET)} trained conditions; one-sided p, shuffled within each model family', f'{r1:+.3f}; p {p1:.4f}', f'>= {CRIT59["h1_rho"]:.2f}; < {CRIT59["h1_p"]}', H1),
    ('H2', f'Spearman rho, change in user drive against change in user instructions followed, {len(H2_SET)} trained conditions; one-sided p, shuffled within each model family', f'{r2:+.3f}; p {p2:.4f}', f'>= {CRIT59["h2_rho"]:.2f}; < {CRIT59["h2_p"]}', H2),
    ('H3', 'Meta-SecAlign-8B: sharedness with its input role minus with the document in the user turn (paired over items)', fmt(h3), 'upper end < 0', H3)]
VT = pd.DataFrame([dict(mark=a, criterion=b, value=v, needed=m_, met=bool(ok_)) for a, b, v, m_, ok_ in checks]); VT.to_csv(f'{REP}/go_criteria_59.csv', index=False)
H3TXT = (f' H3: Meta-SecAlign-8B\'s sharedness is {"lower" if H3 else "not shown lower"} with its input role than with the document in the user turn ({fmt(h3)}; attack success '
         f'{COST["l_sec"]["asr"]:.3f} against {COST["l_sec_user"]["asr"]:.3f}).')
if not V:
    DEC = ('CLOSE: on ' + ', '.join(FAM[f]['label'] for f, v in V_.items() if not vok(v)) + ', an instruction in the document is not treated clearly more like the same instruction from the user than an '
           'ordinary sentence is like itself in the two places, beyond what the same architecture with random weights shows (margin over the highest random draw: ' + '; '.join(f'{FAM[f]["label"]} {v:+.3f}' for f, v in V_.items())
           + '). With the wording taken out, the shared direction of notebooks 57f and 58 does not hold there, and nothing else is read.' + H3TXT)
elif H1 and H2:
    DEC = (f'GO: sharedness tracks what each trained defence achieves (rho {r1:+.2f}, p {p1:.4f}, {len(H1_SET)} conditions) and the user drive tracks what it costs (rho {r2:+.2f}, '
           f'p {p2:.4f}, {len(H2_SET)} conditions). The next notebook trains against sharedness while holding the user drive.' + H3TXT)
elif H1:
    DEC = (f'PARTIAL: sharedness tracks what each trained defence achieves (rho {r1:+.2f}, p {p1:.4f}), but the user drive does not track what it costs (rho {r2:+.2f}, p {p2:.4f}). '
           f'Sharedness is the training target; the user cost has to be measured, not predicted.' + H3TXT)
else:
    DEC = (f'CLOSE: sharedness does not track what the trained defences achieve (rho {r1:+.2f}, p {p1:.4f}, {len(H1_SET)} conditions; user side rho {r2:+.2f}, p {p2:.4f}). '
           f'The shared direction is real in the undefended models but is not what these defences change, and this route gives part two no method.' + H3TXT)
GTXT = f'{DEC}\nV {"pass" if V else "fail"}, H1 {"pass" if H1 else "fail"}, H2 {"pass" if H2 else "fail"}, H3 {"pass" if H3 else "fail"}'
open(f'{REP}/go_59.txt', 'w').write(GTXT + '\n')

# ---- figure ----
COL = {'q3': 'tab:blue', 'q7': 'tab:orange', 'll': 'tab:green'}
fig, ax = plt.subplots(1, 3, figsize=(17, 5))
for k_, (S_, xs, ys, xl, yl) in enumerate(((H1_SET, x1, y1, 'change in sharedness', 'change in attack success (13 fixed tries, paper rule)'),
                                           (H2_SET, x2, y2, 'change in user drive', 'change in user instructions followed (150 twins)'))):
    for t, xv, yv in zip(S_, xs, ys):
        ax[k_].scatter(xv, yv, c=COL[CD[t]['fam']], s=36); ax[k_].annotate(t, (xv, yv), fontsize=7, xytext=(3, 3), textcoords='offset points')
    ax[k_].axhline(0, c='grey', lw=.6); ax[k_].axvline(0, c='grey', lw=.6); ax[k_].set_xlabel(xl); ax[k_].set_ylabel(yl)
    ax[k_].set_title(f'rho {(r1, r2)[k_]:+.2f}, p {(p1, p2)[k_]:.3f}', fontsize=10)
for f in FAM:
    sub_ = LY[LY.model == CD[BASE_OF[f]]['label']]; ax[2].plot(sub_.depth, sub_.sharedness, c=COL[f], label=FAM[f]['label']); ax[2].plot(sub_.depth, sub_.sharedness_random_weights, c=COL[f], ls=':', label=f'{FAM[f]["label"]}, random weights')
ax[2].axhline(0, c='grey', lw=.6); ax[2].set_xlabel('layer (share of depth)'); ax[2].set_ylabel('sharedness within the layer'); ax[2].legend(fontsize=8); ax[2].set_title('undefended models against random weights: sharedness by layer', fontsize=10)
for a_ in ax: a_.grid(alpha=.3)
plt.tight_layout(); plt.savefig(f'{FIG}/drive.png', dpi=150, bbox_inches='tight'); plt.show()
print('\n=== MARKS ==='); print(VT.to_string(index=False)); print('\n' + GTXT)
log_result('nb59: the shared direction across the trained defences: sharedness and user drive on 19 conditions against what each defence achieves and costs',
           '\n\n'.join([GTXT, 'Marks:\n' + VT.to_string(index=False), 'Per condition:\n' + DRT.to_string(index=False)]), reports_dir=REP)

In [ ]:
# === STANDARD COMMIT CELL ===
import os, subprocess
os.chdir(REPO_DIR)
branch = subprocess.run(["git", "rev-parse", "--abbrev-ref", "HEAD"], capture_output=True, text=True).stdout.strip()
assert branch == "main", f"the repository is on branch {branch!r}, not main: switch to main before committing"
r = subprocess.run(["python", "tools/commit_cell.py", "nb59 (follow-on work): the shared direction with the wording taken out, across the trained defences. Notebooks 57f and 58 compared prompts with and without an instruction whose words were largely the same in both directions, which a model with no notion of instructions also responds to alike; this notebook reads, for each of the 210 injections, five prompts (the clean document; the instruction or an ordinary sentence of about the same length on the document's last line; the same two in the user turn) over the five template positions and 16 shared answer tokens, and measures sharedness (per item, the cosine between the change the instruction makes in the document and in the user turn, minus the same for the ordinary sentence) and the user drive (along a user-instruction direction built in the base model from the other half of the hosts and the other six kinds of task), on 19 conditions: Qwen 3B undefended and nine trained versions, Qwen 7B undefended and three, Llama 8B undefended, our two phases, and Meta-SecAlign-8B with its input role and with the document in the user turn; five random-weight draws of each base model's architecture read as the wording null; src/shared_drive.py written here, checked in the notebook against the reading of notebooks 56 to 58; marks V (sharedness of each undefended model against its five random-weight draws), H1 (change in sharedness against change in attack success under the paper's rule, 16 conditions, shuffled within model families), H2 (change in user drive against change in user instructions followed, 15 conditions, shuffled within model families) and H3 (Meta-SecAlign, input role against user turn) fixed in Stage A; costs read from the repository's tables; reports/shared_drive/drive.csv, layers.csv, statements.csv, go_criteria_59.csv, go_59.txt, figures/shared_drive/drive.png"], capture_output=True, text=True)
print(r.stdout); print(r.stderr)